# CDEaDPP 2026.2 - Trabalho Final: Zircônia

## 1. Delimitação do Problema e Desenho da Pesquisa

Este notebook é um **template educacional de protocolo empírico replicável**.
O caso didático é inteiramente simulado e toma como cenário o país imaginário
Zircônia: população grande, fortemente desigual, sistema público de saúde
pressionado por filas e presença territorial incompleta da Defensoria Pública.
O objetivo é ensinar como passar de um problema empírico delimitado para um
quadro de variáveis, dados simulados, exploração, modelagem, testes de
hipóteses e planejamento amostral.

A unidade de análise será o **pedido individual de acesso à saúde** registrado
no ZUS. O desenho é **observacional retrospectivo**: a assistência jurídica
não é sorteada pelo pesquisador; ela ocorre de modo associado a características
da pessoa, do caso e da unidade territorial. Por isso, mesmo quando a simulação
injeta um efeito gerador conhecido, a interpretação estatística no notebook
deve falar em associação ajustada, não em prova causal automática.

| Elemento T.E.M.P. | Delimitação didática | Decisão que o estudante deve adaptar |
|---|---|---|
| Tempo | Pedidos simulados entre 2023 e 2025 | Fixar janela de coleta e seguimento |
| Espaço | 20 regiões e unidades territoriais do ZUS na Zircônia | Descrever sistema de referência e cobertura |
| Matéria | Acesso à saúde pública e acesso à justiça | Definir política, direito ou serviço estudado |
| População | Pessoas de baixa renda com pedido de atendimento/intervenção em saúde | Definir população-alvo, acessível e amostrada |
| Unidade elementar | Pedido administrativo/judicial de saúde | Verificar múltiplos pedidos por pessoa |
| Exposição | Assistência jurídica institucional antes da decisão final | Definir momento e critério de exposição |
| Desfecho primário | Índice de acesso efetivo à saúde | Justificar constructo e escala |
| Desfecho secundário | Desfecho do pedido em cinco categorias nominais | Fixar categorias exaustivas e exclusivas |

**Problema de pesquisa modelo:** entre pedidos de saúde da população de baixa
renda da Zircônia, há associação entre assistência jurídica institucional e
acesso efetivo à saúde, após ajuste por gravidade clínica, renda, idade,
completude documental, região, ano e unidade do ZUS?

**Hipótese primária bilateral:** $H_0: \beta_{\text{assistência jurídica}}=0$
contra $H_1: \beta_{\text{assistência jurídica}}\ne0$ no modelo de regressão
linear para `indice_acesso_saude`.

**Hipótese secundária:** $H_0$: a distribuição de `desfecho_pedido` é igual
entre pedidos com e sem assistência jurídica. $H_1$: ao menos uma das cinco
proporções difere.

**Variáveis ocultas e confundidoras.** Letramento digital, capacidade familiar
de navegar no ZUS, qualidade prévia dos documentos, distância até a Defensoria
e capital social podem afetar simultaneamente a chance de receber assistência
jurídica e o acesso efetivo à saúde. O gerador de dados incluirá uma variável
latente de letramento digital para ilustrar confundimento residual; ela não
será exportada para a base analítica.

**Fontes de apoio usadas na construção do template.** A organização segue os
guias locais da disciplina sobre formulação de problema T.E.M.P., população,
sistema de referência, unidade amostral, escalas de medição, erros amostrais
e não amostrais, hipóteses, erros tipo I e II, validade interna, validade
externa e variáveis ocultas. Como referências bibliográficas de apoio, o
estudante deve consultar, entre outras, Bolfarine e Bussab para amostragem,
Barbetta e Agresti/Finlay para estatística social, Babbie e Bryman para
desenho de pesquisa social, Pearl/Glymour/Jewell para caminhos causais e
Bruce/Bruce para práticas de ciência de dados.

In [1]:
# Importar bibliotecas necessárias para todo o notebook.
import importlib.metadata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
from IPython.display import display
from statsmodels.stats.power import GofChisquarePower, TTestIndPower

# Fixar parâmetros inferenciais antes de observar qualquer resultado.
SEMENTE = 2026
alpha = 0.05
beta = 0.15
poder_alvo = 1 - beta
tamanho_piloto = 50

# Criar gerador principal de números pseudoaleatórios para a simulação.
rng = np.random.default_rng(SEMENTE)

# Localizar a raiz do projeto por arquivos estáveis do repositório.
candidatos_raiz = (Path.cwd(), *Path.cwd().parents)
raizes_possiveis = [
    pasta
    for pasta in candidatos_raiz
    if (pasta / "pyproject.toml").is_file()
    and (pasta / "md" / "trabalho_final_simulacao.md").is_file()
]
if not raizes_possiveis:
    raise FileNotFoundError(
        "Execute o notebook na raiz do projeto ou em uma subpasta dele."
    )
RAIZ_PROJETO = raizes_possiveis[0].resolve()

# Preparar diretórios de saída usados pelas próximas seções.
PASTA_SAIDA = (RAIZ_PROJETO / "out" / "cdedpp_2026_zirconia").resolve()
PASTA_FIGURAS = (PASTA_SAIDA / "figuras").resolve()
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

# Validar parâmetros que controlam inferência e reprodutibilidade.
assert isinstance(SEMENTE, int), "A semente precisa ser um inteiro."
assert 0 < alpha < 1, "alpha deve estar entre 0 e 1."
assert 0 < beta < 1, "beta deve estar entre 0 e 1."
assert np.isclose(poder_alvo, 1 - beta), "poder_alvo deve ser igual a 1 - beta."
assert tamanho_piloto == 50, "Este template exige piloto simulado com n = 50."
assert PASTA_SAIDA.is_relative_to(RAIZ_PROJETO), "Saída fora da raiz do projeto."
assert PASTA_FIGURAS.is_relative_to(PASTA_SAIDA), "Figuras fora da pasta da seção."

# Configurar estilo visual e formato de exibição para tabelas.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 80)

# Registrar versões para reprodutibilidade computacional.
pacotes = ("numpy", "pandas", "matplotlib", "seaborn", "scipy", "statsmodels")
versoes = {
    pacote: importlib.metadata.version(pacote)
    for pacote in pacotes
}
quadro_ambiente = pd.DataFrame(
    {
        "item": ["raiz", "pasta_saida", "semente", "alpha", "beta", "poder", "piloto_n"],
        "valor": [
            str(RAIZ_PROJETO),
            str(PASTA_SAIDA),
            SEMENTE,
            alpha,
            beta,
            poder_alvo,
            tamanho_piloto,
        ],
    }
)

# Exibir o registro do ambiente sem gravar outputs no arquivo-fonte.
display(quadro_ambiente)
display(pd.DataFrame(versoes.items(), columns=["pacote", "versao"]))

,item,valor
0,raiz,E:\DADOS\Documents\AmbienteDeTrabalho\Projetos...
1,pasta_saida,E:\DADOS\Documents\AmbienteDeTrabalho\Projetos...
2,semente,2026
3,alpha,0.05
4,beta,0.15
5,poder,0.85
6,piloto_n,50


,pacote,versao
0,numpy,2.5.2
1,pandas,3.0.5
2,matplotlib,3.11.1
3,seaborn,0.13.2
4,scipy,1.18.0
5,statsmodels,0.14.6


### Interpretação metodológica da seção

A semente garante que os mesmos dados simulados possam ser recuperados por
outro pesquisador. Ela não transforma o desenho em experimento aleatorizado.
O grupo com assistência jurídica continuará sendo selecionado por um mecanismo
observacional, possivelmente relacionado a gravidade, vulnerabilidade, região
e variáveis não observadas. Essa distinção é central para a validade interna.

Ao adaptar o template, o estudante deve registrar: população-alvo, população
acessível, sistema de referência, critério de inclusão, critério de exclusão,
unidade de análise, janela temporal, instrumento de medição, variáveis ocultas
plausíveis, direção esperada dos vieses e limites de generalização.

---

## 2. Dicionário de Dados e Quadro de Variáveis

A tabela abaixo é parte do protocolo, não apenas documentação posterior. Em
uma pesquisa empírica replicável, cada variável deve ter definição operacional,
domínio admissível, escala de mensuração, unidade de medida, tipo computacional
e papel analítico definidos antes da coleta ou da simulação principal.

| Nome abreviado | Significado | Tipo | Escala de medição | Unidade de medida | Tipo Python adequado | Papel | Exemplos |
|---|---|---|---|---|---|---|---|
| `id_pedido` | Identificador artificial único do pedido de acesso à saúde | Texto | Identificador sem escala inferencial | Não se aplica | `string` | Chave | `ZIR-000001`, `ZIR-000250` |
| `assistencia_juridica` | Recebeu assistência jurídica institucional antes da decisão final | Categórica binária | Nominal binária | Categoria Sim/Não | `category` | Independente principal | `Sim`, `Não` |
| `indice_acesso_saude` | Escore composto de acesso efetivo ao cuidado em saúde | Numérica contínua | Intervalar assumida | Pontos de 0 a 100 | `float64` | Dependente primária | `41.7`, `68.2` |
| `desfecho_pedido` | Resultado final observado na janela retrospectiva | Categórica multinomial | Nominal com cinco níveis | Categoria | `category` | Dependente secundária | `internacao_realizada`, `negado` |
| `tempo_espera_dias` | Tempo entre o protocolo do pedido e a decisão ou atendimento | Numérica contínua | Razão | Dias | `float64` | Desfecho auxiliar/covariável descritiva | `12.5`, `83.0` |
| `gravidade_clinica` | Gravidade clínica registrada antes da assistência | Categórica ordinal | Ordinal | Nível 1 a 4 | `Int64` ou `category` ordenada | Confundidora observada | `1`, `4` |
| `renda_pc_zirc` | Renda familiar mensal per capita do requerente | Numérica contínua | Razão | Z$ por pessoa/mês | `float64` | Confundidora observada | `180.50`, `940.00` |
| `idade` | Idade do requerente no protocolo | Numérica discreta | Razão | Anos completos | `Int64` | Confundidora observada | `34`, `72` |
| `regiao_zirconia` | Região administrativa da Zircônia | Categórica multinomial | Nominal | Região | `category` | Estrato/contexto | `R03`, `R17` |
| `unidade_zus` | Unidade territorial do ZUS vinculada ao pedido | Categórica multinomial | Nominal | Unidade administrativa | `category` | Cluster inferencial | `UZ-014`, `UZ-087` |
| `capilaridade_defensoria` | Presença local da Defensoria Pública antes do pedido | Categórica ordinal | Ordinal | Nível 0 a 3 | `Int64` ou `category` ordenada | Confundidora/contexto | `0`, `3` |
| `completude_documental` | Documentação inicial suficiente para análise do pedido | Categórica binária | Nominal binária | Categoria Sim/Não | `category` | Covariável prévia | `Sim`, `Não` |
| `ano_pedido` | Ano de protocolo do pedido | Temporal discreta | Calendário | Ano | `Int64` | Contexto temporal | `2023`, `2025` |
| `letramento_digital_latente` | Facilidade não observada de usar sistemas do ZUS e buscar canais institucionais | Latente binária | Nominal binária | Não observada | Não exportar | Confundidora oculta | `0`, `1` apenas no gerador |

**Ressalva sobre escala intervalar.** O `indice_acesso_saude` será tratado
como intervalar porque diferenças de pontos serão interpretadas como
comparáveis. Isso é uma hipótese substantiva e psicométrica do instrumento,
não uma consequência automática do uso de números decimais. Em dados reais,
o estudante deve justificar itens, pesos, consistência e evidências de validade.

**Ressalva sobre escala multinomial.** `desfecho_pedido` terá cinco categorias
nominais exaustivas e exclusivas. A ordem de apresentação é didática e não
autoriza calcular médias dos códigos nem tratar o desfecho como ordinal sem
argumento teórico independente.

In [2]:
# Definir categorias nominais do desfecho antes da simulação.
CATEGORIAS_DESFECHO = [
    "internacao_realizada",
    "tratamento_ambulatorial",
    "fornecimento_medicamento",
    "negado",
    "pendente_ou_sem_execucao",
]

# Definir categorias binárias padronizadas para variáveis Sim/Não.
CATEGORIAS_BINARIAS = ["Não", "Sim"]

# Listar covariáveis medidas antes da exposição ou no contexto inicial.
COVARIAVEIS_PREVIAS = [
    "gravidade_clinica",
    "renda_pc_zirc",
    "idade",
    "regiao_zirconia",
    "unidade_zus",
    "capilaridade_defensoria",
    "completude_documental",
    "ano_pedido",
]

# Listar variáveis que devem existir na base analítica exportada.
VARIAVEIS_ANALITICAS = [
    "id_pedido",
    "assistencia_juridica",
    "indice_acesso_saude",
    "desfecho_pedido",
    "tempo_espera_dias",
    *COVARIAVEIS_PREVIAS,
]

# Registrar o dicionário também como DataFrame para auditoria computacional.
registros_dicionario = [
    {
        "nome": "id_pedido",
        "significado": "Identificador artificial único do pedido de acesso à saúde",
        "tipo": "texto",
        "escala": "identificador",
        "unidade": "não se aplica",
        "tipo_python": "string",
        "papel": "chave",
        "exemplos": "ZIR-000001; ZIR-000250",
    },
    {
        "nome": "assistencia_juridica",
        "significado": "Recebeu assistência jurídica institucional antes da decisão final",
        "tipo": "categórica binária",
        "escala": "nominal binária",
        "unidade": "Sim/Não",
        "tipo_python": "category",
        "papel": "independente principal",
        "exemplos": "Sim; Não",
    },
    {
        "nome": "indice_acesso_saude",
        "significado": "Escore composto de acesso efetivo ao cuidado em saúde",
        "tipo": "numérica contínua",
        "escala": "intervalar assumida",
        "unidade": "pontos de 0 a 100",
        "tipo_python": "float64",
        "papel": "dependente primária",
        "exemplos": "41.7; 68.2",
    },
    {
        "nome": "desfecho_pedido",
        "significado": "Resultado final observado na janela retrospectiva",
        "tipo": "categórica multinomial",
        "escala": "nominal com cinco níveis",
        "unidade": "categoria",
        "tipo_python": "category",
        "papel": "dependente secundária",
        "exemplos": "internacao_realizada; negado",
    },
    {
        "nome": "tempo_espera_dias",
        "significado": "Tempo entre protocolo e decisão ou atendimento",
        "tipo": "numérica contínua",
        "escala": "razão",
        "unidade": "dias",
        "tipo_python": "float64",
        "papel": "desfecho auxiliar/covariável descritiva",
        "exemplos": "12.5; 83.0",
    },
    {
        "nome": "gravidade_clinica",
        "significado": "Gravidade clínica registrada antes da assistência",
        "tipo": "categórica ordinal",
        "escala": "ordinal",
        "unidade": "nível 1 a 4",
        "tipo_python": "Int64 ou category ordenada",
        "papel": "confundidora observada",
        "exemplos": "1; 4",
    },
    {
        "nome": "renda_pc_zirc",
        "significado": "Renda familiar mensal per capita do requerente",
        "tipo": "numérica contínua",
        "escala": "razão",
        "unidade": "Z$ por pessoa/mês",
        "tipo_python": "float64",
        "papel": "confundidora observada",
        "exemplos": "180.50; 940.00",
    },
    {
        "nome": "idade",
        "significado": "Idade do requerente no protocolo",
        "tipo": "numérica discreta",
        "escala": "razão",
        "unidade": "anos completos",
        "tipo_python": "Int64",
        "papel": "confundidora observada",
        "exemplos": "34; 72",
    },
    {
        "nome": "regiao_zirconia",
        "significado": "Região administrativa da Zircônia",
        "tipo": "categórica multinomial",
        "escala": "nominal",
        "unidade": "região",
        "tipo_python": "category",
        "papel": "estrato/contexto",
        "exemplos": "R03; R17",
    },
    {
        "nome": "unidade_zus",
        "significado": "Unidade territorial do ZUS vinculada ao pedido",
        "tipo": "categórica multinomial",
        "escala": "nominal",
        "unidade": "unidade administrativa",
        "tipo_python": "category",
        "papel": "cluster inferencial",
        "exemplos": "UZ-014; UZ-087",
    },
    {
        "nome": "capilaridade_defensoria",
        "significado": "Presença local da Defensoria Pública antes do pedido",
        "tipo": "categórica ordinal",
        "escala": "ordinal",
        "unidade": "nível 0 a 3",
        "tipo_python": "Int64 ou category ordenada",
        "papel": "confundidora/contexto",
        "exemplos": "0; 3",
    },
    {
        "nome": "completude_documental",
        "significado": "Documentação inicial suficiente para análise do pedido",
        "tipo": "categórica binária",
        "escala": "nominal binária",
        "unidade": "Sim/Não",
        "tipo_python": "category",
        "papel": "covariável prévia",
        "exemplos": "Sim; Não",
    },
    {
        "nome": "ano_pedido",
        "significado": "Ano de protocolo do pedido",
        "tipo": "temporal discreta",
        "escala": "calendário",
        "unidade": "ano",
        "tipo_python": "Int64",
        "papel": "contexto temporal",
        "exemplos": "2023; 2025",
    },
    {
        "nome": "letramento_digital_latente",
        "significado": "Facilidade não observada de usar sistemas do ZUS e buscar canais institucionais",
        "tipo": "latente binária",
        "escala": "nominal binária",
        "unidade": "não observada",
        "tipo_python": "não exportar",
        "papel": "confundidora oculta",
        "exemplos": "0; 1 apenas no gerador",
    },
]
dicionario = pd.DataFrame(registros_dicionario)

# Validar unicidade e completude do dicionário.
assert dicionario["nome"].is_unique, "O dicionário contém nomes duplicados."
assert len(CATEGORIAS_DESFECHO) == 5, "O desfecho multinomial deve ter cinco níveis."
assert len(set(CATEGORIAS_DESFECHO)) == 5, "As categorias do desfecho devem ser distintas."
assert set(VARIAVEIS_ANALITICAS).issubset(set(dicionario["nome"])), (
    "Há variável analítica sem descrição no dicionário."
)
assert "assistencia_juridica" in VARIAVEIS_ANALITICAS, "Exposição ausente."
assert "indice_acesso_saude" in VARIAVEIS_ANALITICAS, "Dependente primária ausente."
assert "desfecho_pedido" in VARIAVEIS_ANALITICAS, "Dependente secundária ausente."
assert "letramento_digital_latente" not in VARIAVEIS_ANALITICAS, (
    "A latente não deve constar da base analítica."
)
assert set(COVARIAVEIS_PREVIAS).issubset(set(VARIAVEIS_ANALITICAS)), (
    "Covariáveis prévias devem estar no esquema analítico."
)

# Exigir preenchimento em todos os campos do quadro.
for coluna in dicionario.columns:
    valores = dicionario[coluna].astype(str).str.strip()
    assert valores.ne("").all(), f"Campo vazio encontrado em {coluna}."

# Exibir a versão computacional do dicionário.
display(dicionario)
print("Total de variáveis analíticas:", len(VARIAVEIS_ANALITICAS))
print("Categorias do desfecho:", ", ".join(CATEGORIAS_DESFECHO))

,nome,significado,tipo,escala,unidade,tipo_python,papel,exemplos
0,id_pedido,Identificador artificial único do pedido de ac...,texto,identificador,não se aplica,string,chave,ZIR-000001; ZIR-000250
1,assistencia_juridica,Recebeu assistência jurídica institucional ant...,categórica binária,nominal binária,Sim/Não,category,independente principal,Sim; Não
2,indice_acesso_saude,Escore composto de acesso efetivo ao cuidado e...,numérica contínua,intervalar assumida,pontos de 0 a 100,float64,dependente primária,41.7; 68.2
3,desfecho_pedido,Resultado final observado na janela retrospectiva,categórica multinomial,nominal com cinco níveis,categoria,category,dependente secundária,internacao_realizada; negado
4,tempo_espera_dias,Tempo entre protocolo e decisão ou atendimento,numérica contínua,razão,dias,float64,desfecho auxiliar/covariável descritiva,12.5; 83.0
5,gravidade_clinica,Gravidade clínica registrada antes da assistência,categórica ordinal,ordinal,nível 1 a 4,Int64 ou category ordenada,confundidora observada,1; 4
6,renda_pc_zirc,Renda familiar mensal per capita do requerente,numérica contínua,razão,Z$ por pessoa/mês,float64,confundidora observada,180.50; 940.00
7,idade,Idade do requerente no protocolo,numérica discreta,razão,anos completos,Int64,confundidora observada,34; 72
8,regiao_zirconia,Região administrativa da Zircônia,categórica multinomial,nominal,região,category,estrato/contexto,R03; R17
9,unidade_zus,Unidade territorial do ZUS vinculada ao pedido,categórica multinomial,nominal,unidade administrativa,category,cluster inferencial,UZ-014; UZ-087


Total de variáveis analíticas: 13
Categorias do desfecho: internacao_realizada, tratamento_ambulatorial, fornecimento_medicamento, negado, pendente_ou_sem_execucao


### Interpretação metodológica da seção

O dicionário explicita quais variáveis são observadas e quais pertencem apenas
ao mecanismo didático de simulação. Essa separação evita uma falha comum:
ajustar o modelo por uma variável que, na pesquisa real, não estaria disponível
ou seria medida depois da exposição.

Em uma adaptação com dados reais, o estudante deve acrescentar: fonte de cada
variável, responsável pela codificação, regras para valores ausentes, tratamento
de duplicidades, janela de aferição e justificativa para o papel analítico
assumido. O quadro também deve ser depositado no repositório antes da coleta,
para reduzir graus de liberdade analíticos e aumentar replicabilidade.

---

## 3. Simulação de Dados (Mock Data) e Piloto Independente

Esta seção constrói uma base simulada para um estudo jurídico-institucional
sobre saúde pública e acesso à justiça em Zircônia. A simulação não pretende
reproduzir uma instituição real; ela cria um **ambiente controlado de ensino**
no qual sabemos que a assistência jurídica tem efeito gerador positivo sobre
o índice de acesso à saúde e altera a distribuição dos desfechos.

O mecanismo gerador segue quatro ideias metodológicas:

1. A exposição `assistencia_juridica` é observacional, não randomizada. A
   probabilidade de receber assistência depende de gravidade clínica, renda,
   capilaridade local da Defensoria, completude documental, região, unidade e
   uma variável latente de letramento digital.
2. O desfecho primário `indice_acesso_saude` recebe um efeito principal
   positivo pré-especificado da assistência jurídica, mas também depende de
   covariáveis prévias, heterogeneidade entre unidades e ruído individual.
3. O desfecho secundário `desfecho_pedido` é multinomial nominal com cinco
   categorias. A assistência jurídica desloca as probabilidades em direção a
   maior realização de internação/tratamento/medicamento e menor negação ou
   pendência.
4. Uma variável latente, `letramento_digital_latente`, influencia tanto a
   exposição quanto os resultados, mas não será exportada. Ela ilustra o
   risco de confundimento residual em estudos retrospectivos.

**Caminhos didáticos do gerador:**  
`gravidade_clinica -> assistencia_juridica` e `gravidade_clinica -> resultado`;  
`capilaridade_defensoria -> assistencia_juridica`;  
`letramento_digital_latente -> assistencia_juridica` e
`letramento_digital_latente -> resultado`;  
`assistencia_juridica -> indice_acesso_saude` e
`assistencia_juridica -> desfecho_pedido`.

O piloto terá **n = 50** e será gerado por fluxo aleatório independente. Ele
será usado depois para estimar desvio padrão, proporção de grupos e parâmetros
preliminares de planejamento amostral. A amostra principal será maior para
permitir EDA, regressão com erros agrupados por unidade e testes sobre a
tabela 2x5.

In [3]:
# Definir parâmetros substantivos do mecanismo gerador antes de simular.
PARAMETROS_GERADORES = {
    "n_amostra_principal": 2500,
    "n_unidades_zus": 80,
    "n_regioes": 20,
    "efeito_assistencia_indice": 7.5,
    "efeito_assistencia_logit_desfecho": 0.75,
    "desvio_ruido_individual": 9.0,
    "desvio_efeito_unidade": 3.0,
    "desvio_efeito_regiao": 2.0,
    "seed_piloto": SEMENTE + 101,
    "seed_principal": SEMENTE + 202,
}

# Conferir que o plano de simulação injeta efeito principal não nulo.
assert PARAMETROS_GERADORES["efeito_assistencia_indice"] != 0
assert PARAMETROS_GERADORES["efeito_assistencia_logit_desfecho"] != 0
assert PARAMETROS_GERADORES["n_unidades_zus"] >= 30
assert PARAMETROS_GERADORES["n_regioes"] == 20

# Criar rótulos estáveis de regiões e unidades.
REGIOES_ZIRCONIA = [f"R{i:02d}" for i in range(1, PARAMETROS_GERADORES["n_regioes"] + 1)]
UNIDADES_ZUS = [f"UZ-{i:03d}" for i in range(1, PARAMETROS_GERADORES["n_unidades_zus"] + 1)]
MAPA_UNIDADE_REGIAO = {
    unidade: REGIOES_ZIRCONIA[(indice - 1) % len(REGIOES_ZIRCONIA)]
    for indice, unidade in enumerate(UNIDADES_ZUS, start=1)
}

def logit_inverso(x):
    # Converter preditor linear em probabilidade no intervalo (0, 1).
    return 1 / (1 + np.exp(-x))

def softmax_estavel(matriz):
    # Transformar escores multinomiais em probabilidades por linha.
    matriz = np.asarray(matriz, dtype=float)
    deslocada = matriz - matriz.max(axis=1, keepdims=True)
    exposta = np.exp(deslocada)
    probabilidades = exposta / exposta.sum(axis=1, keepdims=True)
    return probabilidades

def sortear_categorias(probabilidades, categorias, gerador):
    # Sortear uma categoria por linha usando probabilidades já normalizadas.
    acumuladas = np.cumsum(probabilidades, axis=1)
    sorteios = gerador.random(probabilidades.shape[0])
    indices = (sorteios[:, None] > acumuladas).sum(axis=1)
    return np.array(categorias, dtype=object)[indices]

def gerar_pedidos_simulados(n, prefixo_id, seed):
    # Gerar pedidos simulados com covariáveis, exposição e desfechos.
    # Usar gerador local para tornar cada base reproduzível e independente.
    gerador = np.random.default_rng(seed)

    # Sortear unidades com probabilidades desiguais, aproximando demanda heterogênea.
    pesos_unidades = gerador.gamma(shape=2.5, scale=1.0, size=len(UNIDADES_ZUS))
    pesos_unidades = pesos_unidades / pesos_unidades.sum()
    unidade_zus = gerador.choice(UNIDADES_ZUS, size=n, replace=True, p=pesos_unidades)
    regiao_zirconia = np.array([MAPA_UNIDADE_REGIAO[u] for u in unidade_zus], dtype=object)

    # Gerar efeitos compartilhados por região e por unidade para criar dependência intra-cluster.
    efeito_regiao = {
        regiao: valor
        for regiao, valor in zip(
            REGIOES_ZIRCONIA,
            gerador.normal(0, PARAMETROS_GERADORES["desvio_efeito_regiao"], len(REGIOES_ZIRCONIA)),
            strict=True,
        )
    }
    efeito_unidade = {
        unidade: valor
        for unidade, valor in zip(
            UNIDADES_ZUS,
            gerador.normal(0, PARAMETROS_GERADORES["desvio_efeito_unidade"], len(UNIDADES_ZUS)),
            strict=True,
        )
    }
    ruido_regional = np.array([efeito_regiao[r] for r in regiao_zirconia])
    ruido_unidade = np.array([efeito_unidade[u] for u in unidade_zus])

    # Sortear covariáveis prévias coerentes com o cenário de alta desigualdade.
    ano_pedido = gerador.choice([2023, 2024, 2025], size=n, p=[0.31, 0.34, 0.35])
    idade = np.clip(np.round(gerador.gamma(shape=5.5, scale=9.0, size=n) + 18), 18, 95).astype(int)
    renda_pc_zirc = np.clip(gerador.lognormal(mean=np.log(520), sigma=0.65, size=n), 60, 3500)
    capilaridade_defensoria = gerador.choice([0, 1, 2, 3], size=n, p=[0.24, 0.34, 0.27, 0.15])
    gravidade_clinica = gerador.choice([1, 2, 3, 4], size=n, p=[0.20, 0.36, 0.30, 0.14])

    # Criar variáveis prévias binárias com dependência substantiva.
    prob_doc = logit_inverso(
        -0.15
        + 0.55 * (renda_pc_zirc > 550)
        + 0.28 * capilaridade_defensoria
        - 0.22 * (gravidade_clinica >= 3)
    )
    completude_documental = gerador.binomial(1, prob_doc, size=n)

    # Latente usada apenas no gerador, omitida da base analítica exportada.
    prob_letramento = logit_inverso(
        -0.65
        + 0.75 * (renda_pc_zirc > 700)
        + 0.18 * capilaridade_defensoria
        - 0.012 * np.maximum(idade - 55, 0)
    )
    letramento_digital_latente = gerador.binomial(1, prob_letramento, size=n)

    # Gerar exposição observacional; não há alocação forçada 1:1.
    pred_assistencia = (
        -1.05
        + 0.42 * gravidade_clinica
        - 0.00032 * renda_pc_zirc
        + 0.38 * capilaridade_defensoria
        + 0.45 * (1 - completude_documental)
        + 0.36 * letramento_digital_latente
        + 0.12 * (ano_pedido == 2025)
        + 0.06 * ruido_regional
    )
    prob_assistencia = logit_inverso(pred_assistencia)
    assistencia_binaria = gerador.binomial(1, prob_assistencia, size=n)

    # Injetar o efeito principal da assistência no escore de acesso à saúde.
    indice_latente = (
        43
        + PARAMETROS_GERADORES["efeito_assistencia_indice"] * assistencia_binaria
        + 3.2 * gravidade_clinica
        + 4.4 * completude_documental
        + 1.7 * capilaridade_defensoria
        + 2.8 * letramento_digital_latente
        - 0.0032 * np.maximum(renda_pc_zirc - 900, 0)
        - 0.035 * np.maximum(idade - 70, 0)
        + 0.9 * (ano_pedido == 2025)
        + ruido_regional
        + ruido_unidade
        + gerador.normal(0, PARAMETROS_GERADORES["desvio_ruido_individual"], size=n)
    )
    indice_acesso_saude = np.clip(indice_latente, 0, 100)

    # Gerar tempo de espera como variável de razão; valores positivos em dias.
    media_log_tempo = (
        np.log(54)
        + 0.16 * gravidade_clinica
        - 0.18 * assistencia_binaria
        - 0.10 * completude_documental
        - 0.05 * capilaridade_defensoria
        + 0.06 * ruido_unidade
    )
    tempo_espera_dias = np.clip(gerador.lognormal(media_log_tempo, sigma=0.48), 1, 365)

    # Construir escores multinomiais sem impor ordem ao desfecho.
    base_logits = np.column_stack(
        [
            -0.55 + 0.35 * gravidade_clinica + 0.35 * completude_documental,
            0.20 + 0.18 * gravidade_clinica + 0.25 * completude_documental,
            0.05 + 0.20 * letramento_digital_latente + 0.12 * capilaridade_defensoria,
            0.10 - 0.18 * completude_documental - 0.12 * capilaridade_defensoria,
            -0.10 + 0.18 * (1 - completude_documental) - 0.10 * letramento_digital_latente,
        ]
    )
    efeito_multinomial = PARAMETROS_GERADORES["efeito_assistencia_logit_desfecho"] * assistencia_binaria
    deslocamento_assistencia = np.column_stack(
        [
            0.65 * efeito_multinomial,
            0.30 * efeito_multinomial,
            0.24 * efeito_multinomial,
            -0.55 * efeito_multinomial,
            -0.64 * efeito_multinomial,
        ]
    )
    logits_desfecho = base_logits + deslocamento_assistencia
    probabilidades_desfecho = softmax_estavel(logits_desfecho)
    desfecho_pedido = sortear_categorias(
        probabilidades_desfecho,
        CATEGORIAS_DESFECHO,
        gerador,
    )

    # Montar DataFrame completo para validação, incluindo objetos não exportados.
    dados_completos = pd.DataFrame(
        {
            "id_pedido": [f"{prefixo_id}-{i:06d}" for i in range(1, n + 1)],
            "assistencia_juridica": np.where(assistencia_binaria == 1, "Sim", "Não"),
            "indice_acesso_saude": np.round(indice_acesso_saude, 2),
            "desfecho_pedido": desfecho_pedido,
            "tempo_espera_dias": np.round(tempo_espera_dias, 1),
            "gravidade_clinica": gravidade_clinica,
            "renda_pc_zirc": np.round(renda_pc_zirc, 2),
            "idade": idade,
            "regiao_zirconia": regiao_zirconia,
            "unidade_zus": unidade_zus,
            "capilaridade_defensoria": capilaridade_defensoria,
            "completude_documental": np.where(completude_documental == 1, "Sim", "Não"),
            "ano_pedido": ano_pedido,
            "letramento_digital_latente": letramento_digital_latente,
            "prob_assistencia": prob_assistencia,
        }
    )
    probabilidades = pd.DataFrame(
        probabilidades_desfecho,
        columns=[f"prob_{categoria}" for categoria in CATEGORIAS_DESFECHO],
    )
    dados_completos = pd.concat([dados_completos, probabilidades], axis=1)

    # Criar versão analítica sem latente e sem probabilidades internas.
    dados_analiticos = dados_completos[VARIAVEIS_ANALITICAS].copy()

    # Padronizar tipos categóricos para preservar domínios nas próximas seções.
    dados_analiticos["assistencia_juridica"] = pd.Categorical(
        dados_analiticos["assistencia_juridica"],
        categories=CATEGORIAS_BINARIAS,
    )
    dados_analiticos["completude_documental"] = pd.Categorical(
        dados_analiticos["completude_documental"],
        categories=CATEGORIAS_BINARIAS,
    )
    dados_analiticos["desfecho_pedido"] = pd.Categorical(
        dados_analiticos["desfecho_pedido"],
        categories=CATEGORIAS_DESFECHO,
    )
    dados_analiticos["regiao_zirconia"] = dados_analiticos["regiao_zirconia"].astype("category")
    dados_analiticos["unidade_zus"] = dados_analiticos["unidade_zus"].astype("category")

    return dados_analiticos, dados_completos

# Gerar piloto e amostra principal por fluxos independentes.
piloto, piloto_completo = gerar_pedidos_simulados(
    n=tamanho_piloto,
    prefixo_id="PIL",
    seed=PARAMETROS_GERADORES["seed_piloto"],
)
dados, dados_completos_validacao = gerar_pedidos_simulados(
    n=PARAMETROS_GERADORES["n_amostra_principal"],
    prefixo_id="ZIR",
    seed=PARAMETROS_GERADORES["seed_principal"],
)

# Regerar a amostra principal para confirmar reprodutibilidade exata.
dados_replicados, _ = gerar_pedidos_simulados(
    n=PARAMETROS_GERADORES["n_amostra_principal"],
    prefixo_id="ZIR",
    seed=PARAMETROS_GERADORES["seed_principal"],
)
pd.testing.assert_frame_equal(dados, dados_replicados, check_dtype=True)

# Conferir dimensões, IDs e independência entre piloto e amostra principal.
assert len(piloto) == tamanho_piloto, "O piloto deve ter n = 50."
assert len(dados) == PARAMETROS_GERADORES["n_amostra_principal"]
assert piloto["id_pedido"].is_unique, "IDs do piloto devem ser únicos."
assert dados["id_pedido"].is_unique, "IDs da amostra principal devem ser únicos."
assert set(piloto["id_pedido"]).isdisjoint(set(dados["id_pedido"])), (
    "Piloto e amostra principal não podem compartilhar IDs."
)

# Conferir presença dos dois grupos para permitir razão piloto e comparação.
assert set(piloto["assistencia_juridica"].dropna()) == set(CATEGORIAS_BINARIAS), (
    "O piloto precisa conter casos com e sem assistência jurídica."
)
assert set(dados["assistencia_juridica"].dropna()) == set(CATEGORIAS_BINARIAS), (
    "A amostra principal precisa conter casos com e sem assistência jurídica."
)

# Conferir domínios, finitude dos valores e cobertura territorial.
assert set(dados["desfecho_pedido"].cat.categories) == set(CATEGORIAS_DESFECHO)
assert set(dados["desfecho_pedido"].dropna()).issubset(set(CATEGORIAS_DESFECHO))
assert np.isfinite(dados["indice_acesso_saude"]).all(), "Escore com valor não finito."
assert np.isfinite(piloto["indice_acesso_saude"]).all(), "Piloto com escore não finito."
assert dados["unidade_zus"].nunique() >= 30, "Amostra principal deve ter ao menos 30 unidades."
assert dados["regiao_zirconia"].nunique() == 20, "As 20 regiões devem aparecer na amostra."

# Conferir probabilidades internas do gerador antes de descartá-las.
colunas_prob = [f"prob_{categoria}" for categoria in CATEGORIAS_DESFECHO]
probs = dados_completos_validacao[colunas_prob]
assert np.isfinite(probs.to_numpy()).all(), "Probabilidades não finitas."
assert ((probs >= 0) & (probs <= 1)).all().all(), "Probabilidades fora de [0, 1]."
assert np.allclose(probs.sum(axis=1), 1.0), "Probabilidades não somam 1 por linha."

# Garantir que a latente e probabilidades internas não entraram na base analítica.
colunas_proibidas = {"letramento_digital_latente", "prob_assistencia", *colunas_prob}
assert colunas_proibidas.isdisjoint(set(dados.columns)), "Coluna interna vazou para dados."
assert colunas_proibidas.isdisjoint(set(piloto.columns)), "Coluna interna vazou para piloto."

# Exportar bases analíticas em UTF-8 com BOM para melhor abertura em planilhas.
caminho_dados = PASTA_SAIDA / "zirconia_pedidos_simulados.csv"
caminho_piloto = PASTA_SAIDA / "zirconia_piloto_simulado.csv"
dados.to_csv(caminho_dados, index=False, encoding="utf-8-sig")
piloto.to_csv(caminho_piloto, index=False, encoding="utf-8-sig")

# Produzir resumos iniciais para orientar a leitura sem realizar ainda a EDA formal.
resumo_grupos = (
    dados.groupby("assistencia_juridica", observed=False)
    .agg(
        n=("id_pedido", "size"),
        media_indice=("indice_acesso_saude", "mean"),
        dp_indice=("indice_acesso_saude", "std"),
        mediana_espera=("tempo_espera_dias", "median"),
    )
    .reset_index()
)
resumo_piloto = (
    piloto.groupby("assistencia_juridica", observed=False)
    .agg(
        n=("id_pedido", "size"),
        media_indice=("indice_acesso_saude", "mean"),
        dp_indice=("indice_acesso_saude", "std"),
    )
    .reset_index()
)

# Exibir parâmetros e primeiras tabelas de sanidade.
display(pd.DataFrame(PARAMETROS_GERADORES.items(), columns=["parametro", "valor"]))
display(resumo_grupos)
display(resumo_piloto)
print(f"Dados principais gravados em: {caminho_dados}")
print(f"Piloto gravado em: {caminho_piloto}")
print("Linhas da amostra principal:", len(dados))
print("Unidades ZUS observadas:", dados["unidade_zus"].nunique())

,parametro,valor
0,n_amostra_principal,2500.00
1,n_unidades_zus,80.00
2,n_regioes,20.00
3,efeito_assistencia_indice,7.50
4,efeito_assistencia_logit_desfecho,0.75
5,desvio_ruido_individual,9.00
6,desvio_efeito_unidade,3.00
7,desvio_efeito_regiao,2.00
8,seed_piloto,2127.00
9,seed_principal,2228.00


,assistencia_juridica,n,media_indice,dp_indice,mediana_espera
0,Não,863,55.793,10.527,70.1
1,Sim,1637,65.567,10.560,60.7


,assistencia_juridica,n,media_indice,dp_indice
0,Não,17,55.222,10.846
1,Sim,33,63.272,11.176


Dados principais gravados em: E:\DADOS\Documents\AmbienteDeTrabalho\Projetos\notebook\out\cdedpp_2026_zirconia\zirconia_pedidos_simulados.csv
Piloto gravado em: E:\DADOS\Documents\AmbienteDeTrabalho\Projetos\notebook\out\cdedpp_2026_zirconia\zirconia_piloto_simulado.csv
Linhas da amostra principal: 2500
Unidades ZUS observadas: 80


### Interpretação da simulação

A diferença bruta de médias exibida nesta seção deve ser lida apenas como
**checagem de sanidade do gerador**. A seção de EDA fará a descrição formal,
e a seção de modelagem estimará a associação ajustada com erros padrão
agrupados por unidade. Como a assistência não foi sorteada, parte da diferença
entre grupos pode decorrer de seleção observacional.

O piloto n=50 é propositalmente pequeno. Ele será suficiente para demonstrar
como calcular desvio padrão preliminar e razão entre grupos, mas suas
estimativas serão instáveis. Por isso, o dimensionamento amostral posterior
deverá apresentar análise de sensibilidade para proporções de alocação,
desvio padrão e efeito mínimo relevante.

A variável latente foi usada para construir um cenário mais realista de
confundimento residual. Em uma pesquisa real, o estudante deve perguntar:
quais variáveis não observadas favorecem simultaneamente a procura pela
Defensoria e a obtenção de cuidado em saúde? O viés tenderia a superestimar
ou subestimar a associação de interesse? Que desenho alternativo, coleta
adicional ou análise de sensibilidade poderia enfrentar esse problema?

---

## 4. Curadoria e Auditoria dos Dados Simulados

A simulação produziu arquivos CSV, mas o protocolo empírico não deve confiar
automaticamente em arquivos recém-gerados. Esta seção recarrega as bases como
se tivessem sido recebidas por outro pesquisador, confere encoding, esquema,
domínios, duplicidades, valores ausentes, cobertura territorial e coerência
das frequências.

Em pesquisa real, esta etapa corresponde ao **controle de qualidade da base**:
verificar se cada linha representa a unidade de análise definida, se não há
duplicatas não documentadas, se categorias respeitam o dicionário, se valores
quantitativos são finitos e se perdas ou exclusões foram registradas. A regra
deste template é não apagar casos silenciosamente. Qualquer exclusão deve ter
critério substantivo e rastro reproduzível.

**Fluxo de elegibilidade nesta simulação:**

| Etapa | Regra | Resultado esperado |
|---|---|---|
| Arquivos recebidos | Ler CSVs em UTF-8 com BOM compatível com planilhas | Amostra principal e piloto carregados |
| Unidade de análise | Uma linha por pedido | IDs únicos em cada base |
| Separação das bases | Piloto independente da amostra principal | Sem IDs sobrepostos |
| Domínios categóricos | Categorias iguais ao dicionário | Nenhuma categoria fora do domínio |
| Valores ausentes | NA contabilizado por coluna | Nenhum NA estrutural nesta simulação |
| Curadoria | Sem exclusão quando não houver falha | N preservado |

A categoria `pendente_ou_sem_execucao` é um **estado substantivo do desfecho**,
não um valor ausente. Confundir pendência com NA destruiria informação
empírica importante e enviesaria a descrição da política pública.

In [4]:
# Definir caminhos dos arquivos produzidos pela simulação.
caminho_dados = PASTA_SAIDA / "zirconia_pedidos_simulados.csv"
caminho_piloto = PASTA_SAIDA / "zirconia_piloto_simulado.csv"

# Conferir existência antes da leitura para sinalizar ordem incorreta de execução.
assert caminho_dados.is_file(), f"Arquivo principal não encontrado: {caminho_dados}"
assert caminho_piloto.is_file(), f"Arquivo piloto não encontrado: {caminho_piloto}"

# Ler explicitamente com UTF-8 com BOM, preservando identificadores como texto.
dados_lidos = pd.read_csv(
    caminho_dados,
    encoding="utf-8-sig",
    dtype={
        "id_pedido": "string",
        "assistencia_juridica": "string",
        "desfecho_pedido": "string",
        "regiao_zirconia": "string",
        "unidade_zus": "string",
        "completude_documental": "string",
    },
)
piloto_lido = pd.read_csv(
    caminho_piloto,
    encoding="utf-8-sig",
    dtype={
        "id_pedido": "string",
        "assistencia_juridica": "string",
        "desfecho_pedido": "string",
        "regiao_zirconia": "string",
        "unidade_zus": "string",
        "completude_documental": "string",
    },
)

# Conferir se as colunas correspondem exatamente ao esquema analítico.
colunas_esperadas = list(VARIAVEIS_ANALITICAS)
assert list(dados_lidos.columns) == colunas_esperadas, "Colunas da base principal fora do esquema."
assert list(piloto_lido.columns) == colunas_esperadas, "Colunas do piloto fora do esquema."

# Converter tipos de forma controlada, preservando categorias do dicionário.
def aplicar_esquema(df):
    # Trabalhar em cópia para não alterar o objeto original sem rastreio.
    curado = df.copy()

    # Converter variáveis categóricas para domínios fechados.
    curado["assistencia_juridica"] = pd.Categorical(
        curado["assistencia_juridica"],
        categories=CATEGORIAS_BINARIAS,
    )
    curado["completude_documental"] = pd.Categorical(
        curado["completude_documental"],
        categories=CATEGORIAS_BINARIAS,
    )
    curado["desfecho_pedido"] = pd.Categorical(
        curado["desfecho_pedido"],
        categories=CATEGORIAS_DESFECHO,
    )
    curado["regiao_zirconia"] = curado["regiao_zirconia"].astype("category")
    curado["unidade_zus"] = curado["unidade_zus"].astype("category")

    # Converter variáveis numéricas para os tipos usados nas análises.
    numericas_float = ["indice_acesso_saude", "tempo_espera_dias", "renda_pc_zirc"]
    numericas_int = ["gravidade_clinica", "idade", "capilaridade_defensoria", "ano_pedido"]
    for coluna in numericas_float:
        curado[coluna] = pd.to_numeric(curado[coluna], errors="coerce").astype("float64")
    for coluna in numericas_int:
        curado[coluna] = pd.to_numeric(curado[coluna], errors="coerce").astype("Int64")

    return curado

dados_curados = aplicar_esquema(dados_lidos)
piloto_curado = aplicar_esquema(piloto_lido)

# Montar tabela de tipos para auditoria visual.
tabela_tipos = pd.DataFrame(
    {
        "coluna": dados_curados.columns,
        "tipo_principal": [str(dtype) for dtype in dados_curados.dtypes],
        "tipo_piloto": [str(dtype) for dtype in piloto_curado.dtypes],
    }
)

# Contabilizar valores ausentes sem removê-los automaticamente.
auditoria_na = (
    pd.DataFrame(
        {
            "coluna": dados_curados.columns,
            "na_principal": dados_curados.isna().sum().to_numpy(),
            "na_piloto": piloto_curado.isna().sum().to_numpy(),
        }
    )
    .assign(
        pct_na_principal=lambda x: 100 * x["na_principal"] / len(dados_curados),
        pct_na_piloto=lambda x: 100 * x["na_piloto"] / len(piloto_curado),
    )
)

# Conferir duplicatas de identificador em cada base.
duplicatas = pd.DataFrame(
    {
        "base": ["principal", "piloto"],
        "linhas": [len(dados_curados), len(piloto_curado)],
        "ids_unicos": [dados_curados["id_pedido"].nunique(), piloto_curado["id_pedido"].nunique()],
        "ids_duplicados": [
            dados_curados["id_pedido"].duplicated().sum(),
            piloto_curado["id_pedido"].duplicated().sum(),
        ],
    }
)

# Conferir domínios categóricos observados contra o dicionário.
dominios_observados = {
    "assistencia_juridica": sorted(dados_lidos["assistencia_juridica"].dropna().unique().tolist()),
    "completude_documental": sorted(dados_lidos["completude_documental"].dropna().unique().tolist()),
    "desfecho_pedido": sorted(dados_lidos["desfecho_pedido"].dropna().unique().tolist()),
    "gravidade_clinica": sorted(dados_curados["gravidade_clinica"].dropna().unique().tolist()),
    "capilaridade_defensoria": sorted(dados_curados["capilaridade_defensoria"].dropna().unique().tolist()),
    "ano_pedido": sorted(dados_curados["ano_pedido"].dropna().unique().tolist()),
}
tabela_dominios = pd.DataFrame(
    {
        "variavel": list(dominios_observados.keys()),
        "valores_observados": [", ".join(map(str, valores)) for valores in dominios_observados.values()],
    }
)

# Produzir frequências por unidade e ano para verificar cobertura e estratos.
frequencia_unidade = (
    dados_curados["unidade_zus"]
    .value_counts(dropna=False)
    .rename_axis("unidade_zus")
    .reset_index(name="n")
    .sort_values("unidade_zus")
)
frequencia_ano = (
    dados_curados["ano_pedido"]
    .value_counts(dropna=False)
    .rename_axis("ano_pedido")
    .reset_index(name="n")
    .sort_values("ano_pedido")
)
frequencia_regiao = (
    dados_curados["regiao_zirconia"]
    .value_counts(dropna=False)
    .rename_axis("regiao_zirconia")
    .reset_index(name="n")
    .sort_values("regiao_zirconia")
)

# Criar fluxo de elegibilidade e curadoria sem exclusões nesta iteração.
fluxo_curadoria = pd.DataFrame(
    [
        {
            "etapa": "CSV principal lido",
            "n": len(dados_lidos),
            "observacao": "Arquivo gerado pela seção de simulação.",
        },
        {
            "etapa": "Esquema aplicado",
            "n": len(dados_curados),
            "observacao": "Conversão de tipos sem remoção de linhas.",
        },
        {
            "etapa": "Base curada para análise",
            "n": len(dados_curados),
            "observacao": "Nenhuma exclusão substantiva nesta simulação.",
        },
    ]
)

# Conferir preservação de linhas e separação entre bases.
assert len(dados_lidos) == PARAMETROS_GERADORES["n_amostra_principal"]
assert len(piloto_lido) == tamanho_piloto
assert len(dados_curados) == len(dados_lidos), "Curadoria removeu linhas sem registro."
assert len(piloto_curado) == len(piloto_lido), "Curadoria removeu linhas do piloto sem registro."
assert dados_curados["id_pedido"].is_unique, "Há IDs duplicados na base principal."
assert piloto_curado["id_pedido"].is_unique, "Há IDs duplicados no piloto."
assert set(dados_curados["id_pedido"]).isdisjoint(set(piloto_curado["id_pedido"])), (
    "Piloto e principal compartilham IDs."
)

# Conferir ausência de NA estrutural depois da conversão de tipos.
assert auditoria_na["na_principal"].sum() == 0, "Há NA na base principal."
assert auditoria_na["na_piloto"].sum() == 0, "Há NA no piloto."

# Conferir domínios admissíveis de variáveis categóricas e ordinais.
assert set(dados_lidos["assistencia_juridica"].unique()).issubset(set(CATEGORIAS_BINARIAS))
assert set(dados_lidos["completude_documental"].unique()).issubset(set(CATEGORIAS_BINARIAS))
assert set(dados_lidos["desfecho_pedido"].unique()).issubset(set(CATEGORIAS_DESFECHO))
assert set(piloto_lido["desfecho_pedido"].unique()).issubset(set(CATEGORIAS_DESFECHO))
assert set(dados_curados["gravidade_clinica"].dropna().unique()).issubset({1, 2, 3, 4})
assert set(dados_curados["capilaridade_defensoria"].dropna().unique()).issubset({0, 1, 2, 3})
assert set(dados_curados["ano_pedido"].dropna().unique()).issubset({2023, 2024, 2025})

# Conferir limites substantivos das variáveis quantitativas.
assert dados_curados["indice_acesso_saude"].between(0, 100).all()
assert piloto_curado["indice_acesso_saude"].between(0, 100).all()
assert (dados_curados["tempo_espera_dias"] > 0).all()
assert (piloto_curado["tempo_espera_dias"] > 0).all()
assert (dados_curados["renda_pc_zirc"] > 0).all()
assert dados_curados["idade"].between(18, 95).all()

# Conferir cobertura territorial, temporal e soma dos estratos.
assert dados_curados["unidade_zus"].nunique() >= 30
assert dados_curados["regiao_zirconia"].nunique() == 20
assert frequencia_unidade["n"].sum() == len(dados_curados)
assert frequencia_ano["n"].sum() == len(dados_curados)
assert frequencia_regiao["n"].sum() == len(dados_curados)

# Conferir que a categoria substantiva pendente não foi tratada como ausente.
assert "pendente_ou_sem_execucao" in set(dados_curados["desfecho_pedido"].dropna())
assert dados_curados["desfecho_pedido"].isna().sum() == 0

# Conferir que colunas internas do gerador seguem ausentes.
colunas_internas = {
    "letramento_digital_latente",
    "prob_assistencia",
    *[f"prob_{categoria}" for categoria in CATEGORIAS_DESFECHO],
}
assert colunas_internas.isdisjoint(set(dados_curados.columns))
assert colunas_internas.isdisjoint(set(piloto_curado.columns))

# Exibir quadros de auditoria para leitura pelo estudante.
display(fluxo_curadoria)
display(tabela_tipos)
display(auditoria_na)
display(duplicatas)
display(tabela_dominios)
display(frequencia_ano)
display(frequencia_regiao.head(10))
display(frequencia_unidade.head(10))

# Manter nomes canônicos para as próximas seções do notebook.
dados = dados_curados.copy()
piloto = piloto_curado.copy()

print("Curadoria concluída sem exclusões.")
print("Base principal curada:", dados.shape)
print("Piloto curado:", piloto.shape)

,etapa,n,observacao
0,CSV principal lido,2500,Arquivo gerado pela seção de simulação.
1,Esquema aplicado,2500,Conversão de tipos sem remoção de linhas.
2,Base curada para análise,2500,Nenhuma exclusão substantiva nesta simulação.


,coluna,tipo_principal,tipo_piloto
0,id_pedido,string,string
1,assistencia_juridica,category,category
2,indice_acesso_saude,float64,float64
3,desfecho_pedido,category,category
4,tempo_espera_dias,float64,float64
5,gravidade_clinica,Int64,Int64
6,renda_pc_zirc,float64,float64
7,idade,Int64,Int64
8,regiao_zirconia,category,category
9,unidade_zus,category,category


,coluna,na_principal,na_piloto,pct_na_principal,pct_na_piloto
0,id_pedido,0,0,0.0,0.0
1,assistencia_juridica,0,0,0.0,0.0
2,indice_acesso_saude,0,0,0.0,0.0
3,desfecho_pedido,0,0,0.0,0.0
4,tempo_espera_dias,0,0,0.0,0.0
5,gravidade_clinica,0,0,0.0,0.0
6,renda_pc_zirc,0,0,0.0,0.0
7,idade,0,0,0.0,0.0
8,regiao_zirconia,0,0,0.0,0.0
9,unidade_zus,0,0,0.0,0.0


,base,linhas,ids_unicos,ids_duplicados
0,principal,2500,2500,0
1,piloto,50,50,0


,variavel,valores_observados
0,assistencia_juridica,"Não, Sim"
1,completude_documental,"Não, Sim"
2,desfecho_pedido,"fornecimento_medicamento, internacao_realizada..."
3,gravidade_clinica,"1, 2, 3, 4"
4,capilaridade_defensoria,"0, 1, 2, 3"
5,ano_pedido,"2023, 2024, 2025"


,ano_pedido,n
2,2023,794
1,2024,830
0,2025,876


,regiao_zirconia,n
2,R01,147
14,R02,103
18,R03,68
9,R04,115
15,R05,90
12,R06,109
4,R07,137
10,R08,115
19,R09,57
5,R10,137


,unidade_zus,n
5,UZ-001,62
30,UZ-002,32
61,UZ-003,15
27,UZ-004,33
55,UZ-005,18
62,UZ-006,13
4,UZ-007,68
78,UZ-008,3
35,UZ-009,28
19,UZ-010,40


Curadoria concluída sem exclusões.
Base principal curada: (2500, 13)
Piloto curado: (50, 13)


### Interpretação da curadoria

A auditoria confirmou que a base principal e o piloto preservam o número de
linhas esperado, não compartilham identificadores, respeitam os domínios do
dicionário e não contêm valores ausentes estruturais. Em dados reais, raramente
essa etapa será tão limpa. O ponto pedagógico é que cada decisão de limpeza
deve ser visível, justificada e reproduzível.

Observe que `pendente_ou_sem_execucao` permaneceu como categoria legítima do
desfecho. Ela representa um estado institucional relevante: o pedido ainda não
produziu atendimento efetivo ou não teve execução observada na janela definida.
Removê-la como se fosse ausência de informação reduziria artificialmente a
variabilidade do fenômeno.

Para uma pesquisa observacional retrospectiva, a curadoria também deve discutir
validade externa: quais pedidos nunca chegaram ao sistema de referência? Há
pessoas sem acesso digital ou sem documentação mínima que ficaram invisíveis?
A cobertura territorial da Defensoria e do ZUS afeta quem aparece na base?
Essas perguntas importam tanto quanto os asserts computacionais.